## Multi-variate Gaussian distribution

Perhaps the only distribution that is worth knowing and remembering its form is the multivariate Normal as it has widespread applicability in data science.

$$f_{\mathbf X}(x_1,\ldots,x_k) = \frac{\exp\left(-\frac 1 2 ({\mathbf x}-{\boldsymbol\mu})^\mathrm{T}{\boldsymbol\Sigma}^{-1}({\mathbf x}-{\boldsymbol\mu})\right)}{\sqrt{(2\pi)^n|\boldsymbol\Sigma|}}$$
where $\mathbf{x}$ is a real $n$-dimensional column vector and $|\boldsymbol{\Sigma}| \equiv \operatorname{det}\boldsymbol{\Sigma}$ is the determinant of $\boldsymbol{\Sigma}$.

You can [generate](https://www.youtube.com/watch?v=QCqsJVS8p5A) correlated Gaussian distributions from white gaussian random variables as shown below.  This can be useful in multiple settings. Foe example you can synthesize training examples where there is correlation between features. You can also explain what happens in successive layers of a neural network when a correlated input is propagated through.

![correlated-gaussians](images/correlated-gaussians.png)
_Correlated Gaussians_

## Generating correlated Gaussian random variables

The following code simulates and plots a bivariate normal distribution using the parameters from Example 6.6 of the Math for ML book. The covariance matrix introduces correlation between the two variables, and we visualize both the 3D surface and contour plots of the resulting density.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal

plt.style.use("seaborn-v0_8-dark")
plt.rcParams["figure.figsize"] = 14, 6
fig = plt.figure()

# Initializing the random seed
random_seed = 1000

# Covariance values to iterate over
cov_val = [-1]

# Setting mean of the distribution to be at (0, 2)
mean = np.array([0, 2])

# Storing density function values for further analysis
pdf_list = []

for idx, val in enumerate(cov_val):
    # Initializing the covariance matrix
    cov = np.array([[0.3, val], [val, 5]])

    # Generating a Gaussian bivariate distribution
    # with given mean and covariance matrix
    distr = multivariate_normal(cov=cov, mean=mean, seed=random_seed)

    # Generating a meshgrid complacent with the 3-sigma boundary
    mean_a, mean_b = mean[0], mean[1]
    sigma_a, sigma_b = cov[0, 0], cov[1, 1]

    xa = np.linspace(-4 * sigma_a, 4 * sigma_a, num=100)
    xb = np.linspace(-1 * sigma_b, 1.75 * sigma_b, num=100)
    XA, XB = np.meshgrid(xa, xb)

    # Generating the density function for each point in the meshgrid
    pdf = np.zeros(XA.shape)
    for i in range(XA.shape[0]):
        for j in range(XA.shape[1]):
            pdf[i, j] = distr.pdf([XA[i, j], XB[i, j]])

    # Plotting the density function values
    key = 131 + idx
    ax = fig.add_subplot(key, projection="3d")
    ax.plot_surface(XA, XB, pdf, cmap="viridis")
    plt.xlabel(r"$x_a$")
    plt.ylabel(r"$x_b$")
    plt.title(rf"Joint PDF $p(x_a, x_b)$ with $\Sigma_{{ab}}={val}$")
    pdf_list.append(pdf)
    ax.axes.zaxis.set_ticks([])

plt.tight_layout()
plt.show()

# Plotting contour plots
for idx, val in enumerate(pdf_list):
    plt.subplot(1, 3, idx + 1)
    plt.contourf(XA, XB, val, cmap="viridis")
    plt.xlabel(r"$x_a$")
    plt.ylabel(r"$x_b$")
    plt.title(rf"Joint PDF $p(x_a, x_b)$ with $\Sigma_{{ab}}={cov_val[idx]}$")
plt.tight_layout()
plt.show()


## Conditional distribution $p(x_b \mid x_a)$

For a jointly Gaussian vector $\mathbf{x} = (x_a, x_b)^\top$ partitioned with mean $\boldsymbol{\mu} = (\mu_a, \mu_b)^\top$ and covariance

$$
\boldsymbol{\Sigma} = \begin{pmatrix} \Sigma_{aa} & \Sigma_{ab} \\ \Sigma_{ba} & \Sigma_{bb} \end{pmatrix},
$$

Bishop, *Pattern Recognition and Machine Learning* (2006), §2.3.1, eq. (2.81)-(2.82), gives the conditional in closed form. The result is again Gaussian:

$$
p(\mathbf{x}_b \mid \mathbf{x}_a) = \mathcal{N}\!\left(\mathbf{x}_b \,\big|\, \boldsymbol{\mu}_{b\mid a},\; \boldsymbol{\Sigma}_{b\mid a}\right)
$$

with

$$
\boldsymbol{\mu}_{b\mid a} = \boldsymbol{\mu}_b + \boldsymbol{\Sigma}_{ba}\, \boldsymbol{\Sigma}_{aa}^{-1}\, (\mathbf{x}_a - \boldsymbol{\mu}_a),
\qquad
\boldsymbol{\Sigma}_{b\mid a} = \boldsymbol{\Sigma}_{bb} - \boldsymbol{\Sigma}_{ba}\, \boldsymbol{\Sigma}_{aa}^{-1}\, \boldsymbol{\Sigma}_{ab}.
$$

Two facts to notice:

1. The conditional **mean** is a linear (affine) function of the conditioning value $\mathbf{x}_a$: the regression slope is $\boldsymbol{\Sigma}_{ba}\boldsymbol{\Sigma}_{aa}^{-1}$.

   *Why "regression slope"?* This matrix is the multivariate generalization of the ordinary least-squares slope $\beta = \mathrm{Cov}(Y, X)/\mathrm{Var}(X)$ from simple linear regression. The conditional mean $\mathbb{E}[\mathbf{x}_b \mid \mathbf{x}_a]$ is exactly the best linear (in the minimum-mean-squared-error sense) predictor of $\mathbf{x}_b$ given $\mathbf{x}_a$, and for jointly Gaussian variables this best predictor really is linear, so $\boldsymbol{\Sigma}_{ba}\boldsymbol{\Sigma}_{aa}^{-1}$ plays the role of regression coefficients. In the scalar case it reduces to the familiar $\Sigma_{ba}/\Sigma_{aa} = \mathrm{Cov}(x_b, x_a)/\mathrm{Var}(x_a)$.

   *Dimensions.* Let $\mathbf{x}_a \in \mathbb{R}^{p}$ and $\mathbf{x}_b \in \mathbb{R}^{q}$. Then
    - $\boldsymbol{\Sigma}_{ba}$ is $q \times p$ (cross-covariance, rows indexed by $b$, columns by $a$),
    - $\boldsymbol{\Sigma}_{aa}$ is $p \times p$, and so is its inverse,
    - the product $\boldsymbol{\Sigma}_{ba}\,\boldsymbol{\Sigma}_{aa}^{-1}$ is $q \times p$,
    - so it maps the $p$-vector $(\mathbf{x}_a - \boldsymbol{\mu}_a)$ to a $q$-vector shift of $\boldsymbol{\mu}_b$, exactly what the conditional-mean equation requires.

   In our scalar example below, $p = q = 1$ and the slope is a single number.
2. The conditional **covariance** does *not* depend on $\mathbf{x}_a$: every slice of the joint along a fixed value of $x_a$ has the same shape, only its center shifts.

In the scalar case used here ($x_a, x_b \in \mathbb{R}$, $\Sigma_{ab} = \Sigma_{ba}$):

$$
\mu_{b\mid a}(x_a) = \mu_b + \frac{\Sigma_{ab}}{\Sigma_{aa}}\, (x_a - \mu_a),
\qquad
\sigma^2_{b\mid a} = \Sigma_{bb} - \frac{\Sigma_{ab}^{\,2}}{\Sigma_{aa}}.
$$

The plot below evaluates these formulas for the same covariance matrix used above and overlays the conditional density $p(x_b \mid x_a = x_a^\star)$ for several slice values $x_a^\star$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal, norm

mean = np.array([0.0, 2.0])
cov = np.array([[0.3, -1.0], [-1.0, 5.0]])

mu_a, mu_b = mean
S_aa, S_ab = cov[0, 0], cov[0, 1]
S_bb = cov[1, 1]

# Bishop (2.81)-(2.82) specialised to the scalar case, predicting x_b from x_a
def conditional_xb_given_xa(xa_star):
    mu_cond = mu_b + (S_ab / S_aa) * (xa_star - mu_a)
    var_cond = S_bb - (S_ab ** 2) / S_aa
    return mu_cond, var_cond

# Joint density on a grid
xa_grid = np.linspace(-2.0, 2.0, 200)
xb_grid = np.linspace(-3.0, 7.0, 200)
XA, XB = np.meshgrid(xa_grid, xb_grid)
joint = multivariate_normal(mean=mean, cov=cov)
pdf_joint = joint.pdf(np.dstack([XA, XB]))

# Conditional slices to visualise
slice_vals = [-1.0, 0.0, 1.0]
colors = ["tab:red", "tab:green", "tab:blue"]


In [ ]:
fig, (ax_joint, ax_cond) = plt.subplots(1, 2, figsize=(12, 5))

# Left: joint contour with vertical slice lines
ax_joint.contourf(XA, XB, pdf_joint, levels=20, cmap="viridis")
for xa_star, c in zip(slice_vals, colors):
    ax_joint.axvline(xa_star, color=c, linestyle="--", linewidth=1.5,
                     label=fr"$x_a^\star={xa_star:.1f}$")
ax_joint.set_xlabel(r"$x_a$")
ax_joint.set_ylabel(r"$x_b$")
ax_joint.set_title(r"Joint $p(x_a, x_b)$ with conditioning slices")
ax_joint.legend(loc="upper right")

# Right: conditional densities p(x_b | x_a = x_a*)
for xa_star, c in zip(slice_vals, colors):
    mu_c, var_c = conditional_xb_given_xa(xa_star)
    sd_c = np.sqrt(var_c)
    pdf_cond = norm.pdf(xb_grid, loc=mu_c, scale=sd_c)
    ax_cond.plot(xb_grid, pdf_cond, color=c, linewidth=2,
                 label=fr"$x_a^\star={xa_star:.1f}$,  $\mu_{{b\mid a}}={mu_c:.2f}$,  $\sigma_{{b\mid a}}={sd_c:.2f}$")

ax_cond.set_xlabel(r"$x_b$")
ax_cond.set_ylabel(r"$p(x_b \mid x_a = x_a^\star)$")
ax_cond.set_title(r"Conditional $p(x_b \mid x_a)$ from Bishop (2.81)-(2.82)")
ax_cond.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()
